In [12]:
import pandas as pd
import re
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split


In [6]:
data_path_in  = "DataFolder/train.csv"
data_path_out = "DataFolder/train_urls_masked.csv"

df = pd.read_csv(data_path_in)
df.head()


,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0
2,2,Lol. Try appealing the ban and say you won't d...,No legal advice: Do not offer or request legal...,pcmasterrace,Don't break up with him or call the cops. If ...,It'll be dismissed: https://en.wikipedia.org/w...,Where is there a site that still works where y...,Because this statement of his is true. It isn'...,1
3,3,she will come your home open her legs with an...,"No Advertising: Spam, referral links, unsolici...",sex,Selling Tyrande codes for 3€ to paypal. PM. \n...,tight pussy watch for your cock get her at thi...,NSFW(obviously) http://spankbang.com/iy3u/vide...,Good News ::Download WhatsApp 2.16.230 APK for...,1
4,4,code free tyrande --->>> [Imgur](http://i.imgu...,"No Advertising: Spam, referral links, unsolici...",hearthstone,wow!! amazing reminds me of the old days.Well...,seek for lady for sex in around http://p77.pl/...,must be watch movie https://sites.google.com/s...,We're streaming Pokemon Veitnamese Crystal RIG...,1


In [7]:
TEXT_COLS = [
    "body",
    "positive_example_1", "positive_example_2",
    "negative_example_1", "negative_example_2",
]

In [8]:
URL_RE = re.compile(r'((?:https?://|www\.)[^\s<>"\'\]\)]*)', flags=re.IGNORECASE)

def _parse_url(u: str):
    """Return (scheme, netloc, root, tld, n_subdomains, url_len) or None if not parseable."""
    if not isinstance(u, str) or not u:
        return None
    u2 = u if u.lower().startswith(("http://", "https://")) else "http://" + u
    p = urlparse(u2)
    netloc = p.netloc.lower().lstrip("[").rstrip("]")
    if netloc.startswith("www."):
        netloc = netloc[4:]
    parts = [s for s in netloc.split(".") if s]
    if not parts:
        return None
    root = parts[-2] if len(parts) >= 2 else parts[0]
    tld  = parts[-1] if len(parts) >= 2 else ""
    return p.scheme.lower(), netloc, root, tld, max(0, len(parts) - 2), len(u)

def normalize_urls_simple(text: str) -> str:
    """
    Replace all URLs in `text` with [URL_<root>]; if parsing fails for a match, use [URL].
    No extra columns/features; returns a single, masked string.
    """
    if not isinstance(text, str) or not text.strip():
        return text

    urls = URL_RE.findall(text)
    parsed = [p for u in urls if (p := _parse_url(u))]
    # build a replacement dict for exact matches found
    repl = {}
    for u, (scheme, netloc, root, tld, subd, ulen) in zip(urls, parsed):
        repl[u] = f"[URL_{root}]"

    def _repl(m):
        return repl.get(m.group(0), "[URL]")

    return URL_RE.sub(_repl, text)

In [9]:
for col in TEXT_COLS:
    if col in df.columns:
        df[col] = df[col].apply(normalize_urls_simple)

In [10]:
preview_cols = [c for c in TEXT_COLS if c in df.columns]
print(df[preview_cols].head(5).to_string(index=False))

                                                                                                                                                           body                                                                                                                                                                                                                                                                                                                       positive_example_1                                                                                                                                                           positive_example_2                                                                                                                                                                                                                                                                       negative_example_1                                                          

In [11]:
df.to_csv(data_path_out, index=False)
print(f"\nSaved masked dataset to: {data_path_out}")


Saved masked dataset to: DataFolder/train_urls_masked.csv


In [14]:
from sklearn.model_selection import train_test_split

LABEL_COL = "rule_violation"  
VAL_SIZE  = 0.2                
RANDOM_SEED = 42

df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int).clip(0, 1)

train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    stratify=df[LABEL_COL],
    random_state=RANDOM_SEED,
)

print("Train shape:", train_df.shape, " | pos rate:", round(train_df[LABEL_COL].mean(), 4))
print("Val   shape:", val_df.shape,   " | pos rate:", round(val_df[LABEL_COL].mean(), 4))

train_out = "DataFolder/train_masked.csv"
val_out   = "DataFolder/val_masked.csv"
train_df.to_csv(train_out, index=False)
val_df.to_csv(val_out,   index=False)

print(f"Saved:\n  {train_out}\n  {val_out}")


Train shape: (1623, 9)  | pos rate: 0.5083
Val   shape: (406, 9)  | pos rate: 0.5074
Saved:
  DataFolder/train_masked.csv
  DataFolder/val_masked.csv
